# Notebook 02b — Add Cuisine Categories to the Yelp Dataset, Rebuild PCA

`yelp_ml_ready.csv` has ambience/mood attributes but **no cuisine/category signal at
all** — nothing tells the model a restaurant is BBQ, Mexican, Italian, etc. That's
why restaurants named e.g. "Texas BBQ" got no country-leaning recommendation: there
was no restaurant-side feature for any downstream weight matrix to key off of.

The raw Yelp Academic Dataset (`yelp_academic_dataset_business.json`, downloaded
separately from Kaggle) has a `categories` field the earlier cleaning step dropped.
This notebook:
1. Extracts `business_id` + `categories` from the raw business.json
2. Joins it onto the existing 52,268 restaurants by `business_id` (confirmed 100%
   match beforehand)
3. Maps the ~700 raw category tags down to **17 curated cuisine buckets** (hand-picked
   national/regional cuisine tags only — generic venue type like "Bars"/"Nightlife"
   and dish type like "Sandwiches"/"Desserts" are deliberately excluded, since they
   don't carry cuisine-origin signal)
4. Re-fits the Yelp PCA (same recipe as notebook 04: drop near-constant columns →
   standardize → PCA at 85% variance) with cuisine included

Outputs use a `_new` suffix so the currently-working `restaurant_pca.csv` /
`yelp_pca.pkl` / `yelp_scaler.pkl` are left untouched:
- `data/processed/yelp_ml_ready_new.csv`
- `data/processed/yelp_scaler_new.pkl`
- `data/processed/yelp_pca_new.pkl`
- `data/processed/restaurant_pca_new.csv`


In [1]:
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

RAW = r"C:\Users\ryanm\Documents\coding_projects\restaurant_recommendation\data\raw"
PROCESSED = r"C:\Users\ryanm\Documents\coding_projects\restaurant_recommendation\data\processed"

yelp = pd.read_csv(os.path.join(PROCESSED, "yelp_ml_ready.csv"))
print(f"yelp_ml_ready: {yelp.shape}")


yelp_ml_ready: (52268, 27)


## Step 1 — Pull `business_id` + `categories` out of the raw business.json


In [2]:
records = []
with open(os.path.join(RAW, "yelp_academic_dataset_business.json"), encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        records.append((obj["business_id"], obj.get("categories")))

categories_df = pd.DataFrame(records, columns=["business_id", "categories"])
print(f"business.json: {categories_df.shape}")

yelp = yelp.merge(categories_df, on="business_id", how="left")
print(f"After merge, missing categories: {yelp['categories'].isna().sum()} / {len(yelp)}")


business.json: (150346, 2)
After merge, missing categories: 0 / 52268


## Step 2 — Map ~700 raw tags down to 17 curated cuisine buckets

Only national/regional cuisine tags are kept. A restaurant can match more than one
bucket (e.g. "Mexican" + "Tex-Mex"), so this is multi-hot, not one-hot.


In [3]:
CUISINE_GROUPS = {
    "American": ["American (Traditional)", "American (New)", "Pizza"],
    "Southern": ["Southern"],
    "CajunCreole": ["Cajun/Creole"],
    "TexMex": ["Tex-Mex"],
    "Mexican": ["Mexican"],
    "LatinAmerican": ["Latin American"],
    "Italian": ["Italian"],
    "Mediterranean": ["Mediterranean"],
    "Greek": ["Greek"],
    "Chinese": ["Chinese"],
    "Japanese": ["Japanese"],
    "Thai": ["Thai"],
    "Vietnamese": ["Vietnamese"],
    "Indian": ["Indian"],
    "AsianFusion": ["Asian Fusion"],
    "SteakhouseBBQ": ["Steakhouses", "Barbeque"],
    "SushiBars": ["Sushi Bars"],
}

tag_sets = yelp["categories"].fillna("").apply(lambda s: {t.strip() for t in s.split(",")})

for bucket, raw_tags in CUISINE_GROUPS.items():
    raw_tag_set = set(raw_tags)
    yelp[f"Cuisine.{bucket}"] = tag_sets.apply(lambda tags: int(bool(tags & raw_tag_set)))

CUISINE_COLS = [f"Cuisine.{b}" for b in CUISINE_GROUPS]
print("Restaurants matching at least one cuisine bucket:",
      (yelp[CUISINE_COLS].sum(axis=1) > 0).sum(), "/", len(yelp))
print()
print("Restaurants per bucket:")
print(yelp[CUISINE_COLS].sum().sort_values(ascending=False))


Restaurants matching at least one cuisine bucket: 35666 / 52268

Restaurants per bucket:
Cuisine.American         18688
Cuisine.Mexican           4600
Cuisine.Italian           4573
Cuisine.Chinese           3169
Cuisine.SteakhouseBBQ     3053
Cuisine.Japanese          1830
Cuisine.SushiBars         1717
Cuisine.AsianFusion       1547
Cuisine.Mediterranean     1263
Cuisine.Southern           988
Cuisine.Thai               971
Cuisine.CajunCreole        923
Cuisine.TexMex             919
Cuisine.Vietnamese         852
Cuisine.Indian             838
Cuisine.LatinAmerican      810
Cuisine.Greek              779
dtype: int64


In [4]:
yelp = yelp.drop(columns=["categories"])
yelp.to_csv(os.path.join(PROCESSED, "yelp_ml_ready_new.csv"), index=False)
print(f"Saved yelp_ml_ready_new.csv: {yelp.shape}")


Saved yelp_ml_ready_new.csv: (52268, 44)


## Step 3 — Re-fit PCA (same recipe as notebook 04, cuisine columns added)


In [5]:
YELP_CANDIDATES = [
    "Ambience.romantic", "Ambience.divey", "Ambience.classy",
    "Ambience.hipster", "Ambience.trendy", "Ambience.upscale", "Ambience.casual",
    "HasTV", "HappyHour", "RestaurantsGoodForGroups",
    "GoodForMeal.breakfast", "GoodForMeal.brunch",
    "GoodForMeal.latenight", "GoodForMeal.dinner",
    "RestaurantsTableService", "NoiseLevel", "stars",
] + CUISINE_COLS

YELP_FEATURES = [c for c in YELP_CANDIDATES if c in yelp.columns]
df_yelp = yelp[YELP_FEATURES].dropna().copy()
print(f"Shape after dropna: {df_yelp.shape}")


Shape after dropna: (34808, 34)


In [6]:
variance = df_yelp.var()
low_var_cols = variance[variance < 0.01].index.tolist()

if low_var_cols:
    print(f"Dropping near-constant columns: {low_var_cols}")
    df_yelp = df_yelp.drop(columns=low_var_cols)
    YELP_FEATURES = df_yelp.columns.tolist()
else:
    print("No near-constant columns found.")

print(f"Final shape: {df_yelp.shape}")


No near-constant columns found.
Final shape: (34808, 34)


In [7]:
scaler_y = StandardScaler()
Y_scaled = scaler_y.fit_transform(df_yelp)
print(f"Scaled matrix: {Y_scaled.shape}")


Scaled matrix: (34808, 34)


In [8]:
pca_y = PCA()
pca_y.fit(Y_scaled)
cumvar_y = np.cumsum(pca_y.explained_variance_ratio_)
for i, cv in enumerate(cumvar_y, 1):
    print(f"  {i} components -> {cv:.1%} variance")


  1 components -> 7.6% variance
  2 components -> 14.0% variance
  3 components -> 19.3% variance
  4 components -> 24.0% variance
  5 components -> 28.4% variance
  6 components -> 32.7% variance
  7 components -> 36.5% variance
  8 components -> 40.0% variance
  9 components -> 43.5% variance
  10 components -> 46.7% variance
  11 components -> 49.9% variance
  12 components -> 52.9% variance
  13 components -> 55.9% variance
  14 components -> 58.7% variance
  15 components -> 61.5% variance
  16 components -> 64.2% variance
  17 components -> 66.9% variance
  18 components -> 69.5% variance
  19 components -> 72.1% variance
  20 components -> 74.5% variance
  21 components -> 76.9% variance
  22 components -> 79.2% variance
  23 components -> 81.4% variance
  24 components -> 83.6% variance
  25 components -> 85.7% variance
  26 components -> 87.8% variance
  27 components -> 89.8% variance
  28 components -> 91.6% variance
  29 components -> 93.4% variance
  30 components -> 95.1%

In [9]:
pca_y_final = PCA(n_components=0.85)
restaurant_pca_mat = pca_y_final.fit_transform(Y_scaled)

print(f"Using {pca_y_final.n_components_} components "
      f"({sum(pca_y_final.explained_variance_ratio_):.1%} variance retained)")
print(f"restaurant_pca_mat shape: {restaurant_pca_mat.shape}")


Using 25 components (85.7% variance retained)
restaurant_pca_mat shape: (34808, 25)


## Step 4 — What does each component represent? (cuisine loadings highlighted)


In [10]:
loadings_y = pd.DataFrame(
    pca_y_final.components_,
    index=[f"pc{i+1}" for i in range(pca_y_final.n_components_)],
    columns=YELP_FEATURES,
)
print(loadings_y.round(2).to_string())


      Ambience.romantic  Ambience.divey  Ambience.classy  Ambience.hipster  Ambience.trendy  Ambience.upscale  Ambience.casual  HasTV  HappyHour  RestaurantsGoodForGroups  GoodForMeal.breakfast  GoodForMeal.brunch  GoodForMeal.latenight  GoodForMeal.dinner  RestaurantsTableService  NoiseLevel  stars  Cuisine.American  Cuisine.Southern  Cuisine.CajunCreole  Cuisine.TexMex  Cuisine.Mexican  Cuisine.LatinAmerican  Cuisine.Italian  Cuisine.Mediterranean  Cuisine.Greek  Cuisine.Chinese  Cuisine.Japanese  Cuisine.Thai  Cuisine.Vietnamese  Cuisine.Indian  Cuisine.AsianFusion  Cuisine.SteakhouseBBQ  Cuisine.SushiBars
pc1                0.12           -0.04             0.31              0.12             0.25              0.14             0.28   0.09       0.36                      0.23                   0.04                0.14                   0.13                0.42                     0.40        0.07   0.22              0.13              0.08                 0.07           -0.02          

## Step 5 — Save


In [11]:
restaurant_pca_df = pd.DataFrame(
    restaurant_pca_mat,
    columns=[f"pc{i+1}" for i in range(pca_y_final.n_components_)],
)
surviving_idx = df_yelp.index
restaurant_pca_df.insert(0, "business_id", yelp.loc[surviving_idx, "business_id"].values)
restaurant_pca_df.insert(1, "name", yelp.loc[surviving_idx, "name"].values)

restaurant_pca_df.to_csv(os.path.join(PROCESSED, "restaurant_pca_new.csv"), index=False)
joblib.dump(scaler_y, os.path.join(PROCESSED, "yelp_scaler_new.pkl"))
joblib.dump(pca_y_final, os.path.join(PROCESSED, "yelp_pca_new.pkl"))

print(f"restaurant_pca_new.csv   {restaurant_pca_df.shape}")
restaurant_pca_df.head(3)


restaurant_pca_new.csv   (34808, 27)


,business_id,name,pc1,pc2,pc3,pc4,pc5,pc6,pc7,pc8,...,pc16,pc17,pc18,pc19,pc20,pc21,pc22,pc23,pc24,pc25
0,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,-1.257131,-1.103851,-0.571948,-2.392232,3.049318,-3.286140,-2.152636,-0.108715,...,-0.253562,-0.842732,0.845223,0.565253,-0.047462,0.820127,0.566096,-1.113925,0.888544,-0.694472
1,0bPLkL0QhhPO5kt1_EXmNQ,Zio's Italian Market,-1.142849,-0.051365,0.112736,-0.902010,0.289530,-0.399494,0.749871,1.009335,...,-0.987854,0.466960,-0.691286,-0.129753,0.718257,-0.911966,1.251134,-1.712082,1.047263,0.643396
2,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,3.339153,4.719855,-0.057403,-0.277470,-1.429237,0.142179,-2.988632,-0.921177,...,-0.168255,0.183985,-1.824785,0.697584,0.154612,0.399309,-1.025850,2.168902,0.786681,-1.391182
